# Heisenberg Evolution and Operator Entanglement in a Spin Chain using MPO

(Matrix Product Operators) with bond dimension χ controlled by operator entanglement.



In [46]:
using ITensors, ITensorMPS

cutoff = 1E-8
tau = 0.1

0.1

Evolution of an operator in the Heisenberg representation : 
$$O(t)=U^\dagger(t) O U(t)$$

with $U(t) = e^{-iHt}$

---

let's creat a spin chain of $6$, $1/2$ particules.

In [47]:
# nombre de spins
N = 6

# creation des sites
sites = ITensors.siteinds("S=1/2", N)

println(sites)

Index{Int64}[(dim=2|id=454|"S=1/2,Site,n=1"), (dim=2|id=33|"S=1/2,Site,n=2"), (dim=2|id=381|"S=1/2,Site,n=3"), (dim=2|id=674|"S=1/2,Site,n=4"), (dim=2|id=922|"S=1/2,Site,n=5"), (dim=2|id=292|"S=1/2,Site,n=6")]


---

If $O = I$ we have :

$$U^\dagger I U = I \qquad \forall U$$

let's check

In [48]:
# construction MPO identite
IdMPO = MPO(sites, "Id")

6-element MPO:
 ((dim=2|id=454|"S=1/2,Site,n=1")', (dim=2|id=454|"S=1/2,Site,n=1"), (dim=1|id=466|"Link,l=1"))
 ((dim=2|id=33|"S=1/2,Site,n=2")', (dim=2|id=33|"S=1/2,Site,n=2"), (dim=1|id=808|"Link,l=2"), (dim=1|id=466|"Link,l=1"))
 ((dim=2|id=381|"S=1/2,Site,n=3")', (dim=2|id=381|"S=1/2,Site,n=3"), (dim=1|id=690|"Link,l=3"), (dim=1|id=808|"Link,l=2"))
 ((dim=2|id=674|"S=1/2,Site,n=4")', (dim=2|id=674|"S=1/2,Site,n=4"), (dim=1|id=548|"Link,l=4"), (dim=1|id=690|"Link,l=3"))
 ((dim=2|id=922|"S=1/2,Site,n=5")', (dim=2|id=922|"S=1/2,Site,n=5"), (dim=1|id=205|"Link,l=5"), (dim=1|id=548|"Link,l=4"))
 ((dim=2|id=292|"S=1/2,Site,n=6")', (dim=2|id=292|"S=1/2,Site,n=6"), (dim=1|id=205|"Link,l=5"))

In [49]:
println("Structure du MPO identite :")
println(IdMPO)

println("\nBond indices :")
println(linkinds(IdMPO))

Structure du MPO identite :
MPO(6)

Bond indices :
Index{Int64}[(dim=1|id=466|"Link,l=1"), (dim=1|id=808|"Link,l=2"), (dim=1|id=690|"Link,l=3"), (dim=1|id=548|"Link,l=4"), (dim=1|id=205|"Link,l=5")]


The MPO had $6$ Tensors with $N=6$ we are happy.

all have $dim = 1 \implies \chi = 1$, it's the simpliest MPO possible.

---

Here we will take the Hamiltonian :

$$H = \sum_j ( X_j X_{j+1} + gX_j + hZ_j )$$

In [ ]:
g = 0.5
h = 0.5

ampo = AutoMPO()

for j=1:N-1 
    add!(ampo,"X",j,"X",j+1) # interaction XX
end

for j=1:N 
    add!(ampo, g,"X",j) # champ X
    add!(ampo, h,"Z",j) # champ Z
end

H = MPO(ampo, sites)

linkinds(H)

5-element Vector{Index{Int64}}:
 (dim=3|id=43|"Link,l=1")
 (dim=3|id=648|"Link,l=2")
 (dim=3|id=409|"Link,l=3")
 (dim=3|id=419|"Link,l=4")
 (dim=3|id=413|"Link,l=5")

bond dimension faible, $3$.

La matrice complète serait : $2^N × 2^N$

donc pour $N=6$ : $64 × 64$

Mais le MPO reste très compact.

In [65]:
println(norm(H))

8.246211251235321


---

for TEBD

In [ ]:
g, h = 1/2, 1/2

gates = ITensor[]
for j in 1:(N - 1)
    s1 = sites[j]
    s2 = sites[j + 1]
    hj =
      op("X", s1) * op("X", s2) +
      g * op("X", s1) * op("Id", s2) +
      h * op("Z", s1) * op("Id", s2)
    Gj = exp(-im * tau / 2 * hj)
    push!(gates, Gj)
end